<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# Detection de Pneumonie sur Radiographies Thoraciques

**Contexte metier**  
La pneumonie est l'une des principales causes de mortalite infantile dans le monde. Le diagnostic repose sur la lecture de radiographies thoraciques (chest X-ray), une tache chronophage et sujette a variabilite inter-expert. Un modele de detection automatique peut assister le radiologue, reduire les delais de diagnostic et ameliorer la prise en charge dans les environnements sous-dotes en specialistes.

**Objectif**  
Classer automatiquement une radiographie thoracique en : **NORMAL** ou **PNEUMONIA**.

**Dataset**  
Chest X-Ray Images (Pneumonia) - Kaggle / Paul Mooney  
5 863 images JPEG organisees en train / val / test, 2 classes.

**Elements TensorFlow mis en oeuvre**  
tf.data pipeline, augmentation integree, class weights, callbacks, Grad-CAM, sauvegarde .keras

---
**Pipeline** : Collecte -> EDA -> Preprocessing -> Augmentation -> Modelisation -> Evaluation -> Grad-CAM -> Comparaison -> Sauvegarde

## 0. Installation et imports

In [1]:
!pip install -q kagglehub

import os, pathlib, random, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, BatchNormalization,
    Rescaling, RandomFlip, RandomRotation, RandomZoom, RandomContrast
)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint,
    ReduceLROnPlateau, TensorBoard
)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU : {tf.config.list_physical_devices('GPU')}")


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


TensorFlow : 2.21.0


GPU : []


---
## 1. Collecte des donnees

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
import kagglehub

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
DATA_DIR = pathlib.Path(path) / "chest_xray"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

for split, d in [("Train", TRAIN_DIR), ("Val", VAL_DIR), ("Test", TEST_DIR)]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        n = len(list((d / cls).glob("*.jpeg")) + list((d / cls).glob("*.jpg")))
        print(f"  {split:6s} | {cls:9s} : {n} images")

  0%|                                              | 0.00/2.29G [00:00<?, ?B/s]

  0%|                                     | 1.00M/2.29G [00:00<37:18, 1.10MB/s]

  0%|                                     | 2.00M/2.29G [00:01<23:44, 1.73MB/s]

  0%|                                     | 3.00M/2.29G [00:01<18:16, 2.24MB/s]

  0%|                                     | 4.00M/2.29G [00:01<15:45, 2.60MB/s]

...

 50%|██████████████████▍                  | 1.14G/2.29G [06:47<05:57, 3.47MB/s]

 50%|██████████████████▍                  | 1.14G/2.29G [06:47<05:41, 3.62MB/s]

 50%|██████████████████▍                  | 1.14G/2.29G [06:47<05:41, 3.63MB/s]

 50%|██████████████████▍                  | 1.14G/2.29G [06:51<06:54, 2.98MB/s]

ChunkedEncodingError: ("Connection broken: ConnectionAbortedError(10053, 'Une connexion établie a été abandonnée par un logiciel de votre ordinateur hôte', None, 10053, None)", ConnectionAbortedError(10053, 'Une connexion établie a été abandonnée par un logiciel de votre ordinateur hôte', None, 10053, None))

---
## 2. Analyse exploratoire (EDA)

In [ ]:
from PIL import Image as PILImage

counts = {}
for split, d in [("Train", TRAIN_DIR), ("Val", VAL_DIR), ("Test", TEST_DIR)]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        files = list((d / cls).glob("*.jpeg")) + list((d / cls).glob("*.jpg"))
        counts[f"{split}_{cls}"] = len(files)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, split in zip(axes, ["Train", "Val", "Test"]):
    vals  = [counts[f"{split}_NORMAL"], counts[f"{split}_PNEUMONIA"]]
    bars  = ax.bar(["NORMAL", "PNEUMONIA"], vals, color=["steelblue", "tomato"])
    ax.set_title(f"{split} set")
    ax.set_ylabel("Nombre d'images")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 20, str(v), ha='center', fontsize=10)

plt.suptitle("Distribution des classes par split", fontweight='bold')
plt.tight_layout()
plt.show()

total_n = counts['Train_NORMAL']
total_p = counts['Train_PNEUMONIA']
ratio   = total_p / total_n
print(f"Ratio PNEUMONIA/NORMAL (train) : {ratio:.2f}")
print("=> Dataset desequilibre : PNEUMONIA 3x plus representee.")
print("   Correction via class_weight lors de l'entrainement.")

In [ ]:
# Visualisation d'echantillons par classe
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for row, cls in enumerate(["NORMAL", "PNEUMONIA"]):
    files = list((TRAIN_DIR / cls).glob("*.jpeg"))[:5]
    for ax, f in zip(axes[row], files):
        img = PILImage.open(f).convert("L")
        ax.imshow(img, cmap='gray')
        ax.set_title(cls, color='steelblue' if cls=='NORMAL' else 'tomato', fontsize=9)
        ax.axis('off')

plt.suptitle("Exemples de radiographies thoraciques", fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des dimensions d'images
widths, heights = [], []
for cls in ["NORMAL", "PNEUMONIA"]:
    for f in random.sample(list((TRAIN_DIR / cls).glob("*.jpeg")), 100):
        with PILImage.open(f) as img:
            widths.append(img.width)
            heights.append(img.height)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des largeurs (200 images)')
axes[0].set_xlabel('Pixels')
axes[1].hist(heights, bins=30, color='tomato', edgecolor='white')
axes[1].set_title('Distribution des hauteurs')
axes[1].set_xlabel('Pixels')
plt.tight_layout()
plt.show()

print(f"Largeur  : moy={np.mean(widths):.0f}px, min={min(widths)}, max={max(widths)}")
print(f"Hauteur  : moy={np.mean(heights):.0f}px, min={min(heights)}, max={max(heights)}")
print("=> Redimensionnement a 224x224 pour normaliser les entrees.")

---
## 3. Preprocessing et pipeline tf.data

**Choix techniques justifies :**

| Choix | Valeur | Justification |
|---|---|---|
| IMG_SIZE | 224x224 | Standard ImageNet, compatible EfficientNetB0 |
| Color mode | RGB | EfficientNetB0 attend 3 canaux ; les radiographies en niveaux de gris sont repliees sur 3 canaux automatiquement |
| Val split | extrait du train | Le val officiel ne contient que 16 images (inutilisable), on cree un vrai val a 15% du train |
| Class weights | automatique | Corrige le desequilibre PNEUMONIA/NORMAL sans sur-echantillonnage |
| tf.data | cache + prefetch | Supprime le goulot CPU/GPU, chargement en parallele avec AUTOTUNE |

### Justification du Val set custom
Le dataset Kaggle fourni un val set de seulement **16 images** (8 par classe), ce qui est statistiquement inexploitable pour evaluer la generalisation. On reconstruit un split 70/15/15 depuis le train set en conservant la stratification.

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# Chargement train+val depuis le repertoire train (le val officiel est inutilisable)
full_train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=None,
    shuffle=True,
    seed=SEED,
    color_mode='rgb'
)

CLASS_NAMES = full_train_ds.class_names
print(f"Classes : {CLASS_NAMES}")

# Extraction en numpy pour split sklearn
X_all = np.array([x.numpy() for x, _ in full_train_ds])
y_all = np.array([y.numpy() for _, y in full_train_ds])
print(f"Shape : {X_all.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

# Split 70% train / 15% val / 15% test interne (stratifie)
X_tv, X_test_int, y_tv, y_test_int = train_test_split(
    X_all, y_all, test_size=0.15, random_state=SEED, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/0.85, random_state=SEED, stratify=y_tv
)

# Chargement du test set officiel Kaggle
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, labels='inferred', label_mode='int',
    image_size=IMG_SIZE, batch_size=None, shuffle=False, color_mode='rgb'
)
X_test = np.array([x.numpy() for x, _ in test_ds_raw])
y_test = np.array([y.numpy() for _, y in test_ds_raw])

print(f"Train : {X_train.shape[0]} | Val : {X_val.shape[0]} | Test officiel : {X_test.shape[0]}")

# Calcul des class weights pour corriger le desequilibre
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
CLASS_WEIGHTS = {i: cw[i] for i in range(len(cw))}
print(f"Class weights : {CLASS_WEIGHTS}")

In [ ]:
# Pipeline tf.data optimise
def make_dataset(X, y, augment=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((X.astype('float32'), y.astype('int32')))
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).cache().prefetch(AUTOTUNE)
    return ds

# Couches d'augmentation (actives uniquement en training=True)
augmentation_layer = Sequential([
    RandomFlip('horizontal'),
    RandomRotation(0.08),
    RandomZoom(0.10),
    RandomContrast(0.15),
], name='augmentation')

def augment_fn(image, label):
    image = augmentation_layer(image, training=True)
    return image, label

train_ds = make_dataset(X_train, y_train, augment=True)
val_ds   = make_dataset(X_val,   y_val,   augment=False)
test_ds  = make_dataset(X_test,  y_test,  augment=False)

print("Pipelines tf.data prets.")

In [ ]:
# Visualisation de l'augmentation
sample_img = X_train[0]
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0][0].imshow(sample_img.astype('uint8'), cmap='gray')
axes[0][0].set_title('Originale')
axes[0][0].axis('off')

for i in range(1, 10):
    aug_img, _ = augment_fn(sample_img, 0)
    row, col   = divmod(i, 5)
    axes[row][col].imshow(aug_img.numpy().astype('uint8'), cmap='gray')
    axes[row][col].set_title(f'Aug #{i}')
    axes[row][col].axis('off')

plt.suptitle('Effet de la data augmentation', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Modele 1 : CNN Custom (fait maison)

**Justification de l'architecture :**

| Couche | Role |
|---|---|
| `Rescaling(1/255)` | Normalise les pixels en [0,1] a l'interieur du modele |
| `Conv2D(F, 3x3) + BN + MaxPool` | Extraction de features spatiales + stabilisation des gradients + sous-echantillonnage |
| Progression 32->64->128->256 | Features de plus en plus abstraites a chaque bloc |
| `GlobalAveragePooling2D` | Remplace Flatten : 60x moins de parametres, moins d'overfitting |
| `Dropout(0.5)` | Regularisation forte, dataset de taille moderee |
| `Dense(1, sigmoid)` | Probabilite de pneumonie (classification binaire) |

**Particularite metier** : on prefere un **recall eleve** (PNEUMONIA) pour minimiser les faux negatifs (ne pas rater une pneumonie est critique). Le seuil de classification sera ajuste en consequence.

In [ ]:
def build_cnn_custom():
    model = Sequential(name='CNN_Custom', layers=[
        Rescaling(1./255, input_shape=(224, 224, 3)),

        Conv2D(32,  (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),

        Conv2D(64,  (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),

        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),

        Conv2D(256, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),

        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.Precision(name='precision')]
    )
    return model

cnn_custom = build_cnn_custom()
cnn_custom.summary()

In [ ]:
callbacks_cnn = [
    EarlyStopping(monitor='val_auc', patience=8, restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint('best_cnn_custom.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
]

history_cnn = cnn_custom.fit(
    train_ds,
    epochs=40,
    validation_data=val_ds,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_cnn,
    verbose=1
)

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy'), ('auc', 'AUC')]
    for ax, (m, label) in zip(axes, metrics):
        ax.plot(history.history[m],         label='Train')
        ax.plot(history.history[f'val_{m}'], label='Val', linestyle='--')
        ax.set_title(label)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(alpha=0.3)
    plt.suptitle(title, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_history(history_cnn, 'Courbes apprentissage : CNN Custom')

---
## 5. Modele 2 : Transfer Learning EfficientNetB0

**Justification du choix EfficientNetB0 :**

EfficientNet propose une mise a l'echelle composee (largeur + profondeur + resolution) optimisee par NAS (Neural Architecture Search). La version B0 offre le meilleur compromis parametres/performance pour un dataset de taille moderee.

| Critere | EfficientNetB0 | VGG16 | ResNet50 |
|---|---|---|---|
| Parametres | 5.3M | 138M | 25.6M |
| Top-1 ImageNet | 77.1% | 71.3% | 76.0% |
| Vitesse inference | Rapide | Lent | Moyen |
| Adapte dataset moyen | Oui | Non (overfit) | Moyen |

**Strategie en deux phases :**
1. Phase 1 (base gelee) : on entraine uniquement la tete de classification -> convergence rapide
2. Phase 2 (fine-tuning) : on debloque les 50 dernieres couches -> adaptation aux radiographies

In [ ]:
def build_efficientnet(trainable_base=False, unfreeze_from=None):
    base = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    if not trainable_base:
        base.trainable = False
    elif unfreeze_from is not None:
        base.trainable = True
        for layer in base.layers[:unfreeze_from]:
            layer.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x      = base(inputs, training=trainable_base)   # tf.cast supprimé
    x      = GlobalAveragePooling2D()(x)
    x      = Dense(256, activation='relu')(x)
    x      = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='EfficientNetB0_TL')
    model.compile(
        optimizer=Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.Precision(name='precision')]
    )
    trainable_count = sum([np.prod(v.shape) for v in model.trainable_variables])
    total_count     = sum([np.prod(v.shape) for v in model.variables])
    print(f"Params entrainables : {trainable_count:,} / {total_count:,}")
    return model

effnet_frozen = build_efficientnet(trainable_base=False)
effnet_frozen.summary()

In [ ]:
# Phase 1 : entrainement tete uniquement
callbacks_eff1 = [
    EarlyStopping(monitor='val_auc', patience=6, restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint('best_effnet_frozen.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

history_eff1 = effnet_frozen.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_eff1,
    verbose=1
)

plot_history(history_eff1, 'Courbes apprentissage : EfficientNetB0 Phase 1 (base gelee)')

In [ ]:
# Phase 2 : fine-tuning des 50 dernieres couches de la base
# Justification : les 50 dernieres couches capturent des features hauts niveau
# Les adapter aux radiographies (domaine different d'ImageNet) ameliore la performance.
# On utilise un lr 10x plus faible pour ne pas degrader les poids pre-entraines.

effnet_ft = build_efficientnet(trainable_base=True, unfreeze_from=-50)
effnet_ft.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.Precision(name='precision')]
)

# Chargement des meilleurs poids Phase 1
effnet_ft.load_weights('best_effnet_frozen.keras', skip_mismatch=True)

callbacks_eff2 = [
    EarlyStopping(monitor='val_auc', patience=8, restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint('best_effnet_finetuned.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-8, verbose=1)
]

history_eff2 = effnet_ft.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_eff2,
    verbose=1
)

plot_history(history_eff2, 'Courbes apprentissage : EfficientNetB0 Phase 2 (fine-tuning)')

---
## 6. Evaluation des modeles

**Metriques retenues et justification metier :**

| Metrique | Formule | Justification metier |
|---|---|---|
| Accuracy | (TP+TN)/total | Vision globale |
| **Recall (Sensitivity)** | TP/(TP+FN) | **Prioritaire** : ne pas rater une pneumonie = faux negatif inacceptable |
| Precision | TP/(TP+FP) | Eviter les hospitalisations inutiles |
| F1-Score | 2*P*R/(P+R) | Equilibre Precision/Recall |
| ROC-AUC | Aire ROC | Mesure de discrimination independante du seuil |
| Specificite | TN/(TN+FP) | Capacite a identifier correctement les sains |

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, threshold=0.5):
    y_proba = model.predict(X_test / 255.0 if 'Custom' in model_name else X_test,
                            batch_size=32, verbose=0).flatten()
    y_pred  = (y_proba >= threshold).astype(int)

    acc  = (y_pred == y_test).mean()
    auc  = roc_auc_score(y_test, y_proba)
    f1   = f1_score(y_test, y_pred)

    from sklearn.metrics import recall_score, precision_score
    rec  = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    spec = recall_score(y_test, y_pred, pos_label=0)

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  AUC         : {auc:.4f}")
    print(f"  Recall      : {rec:.4f}  <- prioritaire")
    print(f"  Precision   : {prec:.4f}")
    print(f"  F1-Score    : {f1:.4f}")
    print(f"  Specificite : {spec:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

    return {
        'model': model_name, 'accuracy': acc, 'auc': auc,
        'recall': rec, 'precision': prec, 'f1': f1,
        'specificity': spec, 'y_proba': y_proba, 'y_pred': y_pred
    }

results_cnn   = evaluate_model(cnn_custom,  X_test, y_test, 'CNN Custom')
results_eff1  = evaluate_model(effnet_frozen, X_test, y_test, 'EfficientNetB0 Frozen')
results_eff2  = evaluate_model(effnet_ft,   X_test, y_test, 'EfficientNetB0 Fine-tuned')

In [ ]:
# Matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
all_results = [results_cnn, results_eff1, results_eff2]

for ax, res in zip(axes, all_results):
    cm_val = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['NORMAL', 'PNEUMONIA'],
                yticklabels=['NORMAL', 'PNEUMONIA'])
    ax.set_title(res['model'], fontweight='bold')
    ax.set_xlabel('Predit')
    ax.set_ylabel('Reel')

plt.suptitle('Matrices de confusion : Test set officiel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Courbes ROC comparatives
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'tomato', 'seagreen']

for res, color in zip(all_results, colors):
    RocCurveDisplay.from_predictions(
        y_test, res['y_proba'],
        name=f"{res['model']} (AUC={res['auc']:.3f})",
        ax=ax, color=color
    )

ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Aleatoire')
ax.set_title('Courbes ROC : comparaison des modeles', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Ajustement du seuil de decision

**Justification metier** : dans un contexte de diagnostic medical, un faux negatif (pneumonie non detectee) est bien plus grave qu'un faux positif (examen supplementaire prescrit a tort). On abaisse donc le seuil de classification pour maximiser le recall sur la classe PNEUMONIA, au prix d'une precision legerement reduite.

In [ ]:
from sklearn.metrics import precision_recall_curve

best_model_results = results_eff2  # modele selectionne
y_proba_best       = best_model_results['y_proba']

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds, precisions[:-1], label='Precision', color='steelblue')
axes[0].plot(thresholds, recalls[:-1],    label='Recall',    color='tomato')
axes[0].axvline(0.3, color='gray', linestyle='--', label='Seuil=0.3')
axes[0].axvline(0.5, color='black', linestyle='--', label='Seuil=0.5')
axes[0].set_xlabel('Seuil')
axes[0].set_title('Precision et Recall selon le seuil')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1 par seuil
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_t    = thresholds[np.argmax(f1_scores)]
axes[1].plot(thresholds, f1_scores, color='seagreen')
axes[1].axvline(best_t, color='red', linestyle='--', label=f'Seuil optimal F1 = {best_t:.2f}')
axes[1].set_xlabel('Seuil')
axes[1].set_title('F1-Score selon le seuil')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Analyse du seuil de decision : EfficientNetB0 Fine-tuned', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Seuil optimal F1 : {best_t:.3f}")
print("\nEvaluation avec seuil 0.3 (priorite recall) :")
_ = evaluate_model(effnet_ft, X_test, y_test, 'EfficientNetB0 Fine-tuned (seuil=0.3)', threshold=0.3)

---
## 8. Explainabilite : Grad-CAM

**Grad-CAM** (Gradient-weighted Class Activation Mapping) visualise les regions de l'image qui ont contribue le plus a la decision du modele. C'est un outil indispensable en contexte medical pour :
- Valider que le modele regarde bien les poumons et non des artefacts
- Communiquer les resultats aux experts medicaux
- Detecter les biais eventuels du modele

In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(np.expand_dims(img_array, 0))
        loss = predictions[:, 0]

    grads     = tape.gradient(loss, conv_outputs)
    pooled    = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out  = conv_outputs[0]
    heatmap   = conv_out @ pooled[..., tf.newaxis]
    heatmap   = tf.squeeze(heatmap)
    heatmap   = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def display_gradcam(model, img_array, true_label, pred_label, last_conv_layer):
    heatmap = get_gradcam_heatmap(model, img_array, last_conv_layer)

    import cv2
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cm.jet(heatmap_colored)[:, :, :3]

    img_norm = img_array / 255.0
    superimposed = heatmap_colored * 0.4 + img_norm
    superimposed = superimposed / superimposed.max()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_array.astype('uint8'), cmap='gray')
    axes[0].set_title('Image originale')
    axes[0].axis('off')
    axes[1].imshow(heatmap_resized, cmap='jet')
    axes[1].set_title('Heatmap Grad-CAM')
    axes[1].axis('off')
    axes[2].imshow(superimposed)
    axes[2].set_title(f'Reel : {CLASS_NAMES[true_label]}\nPredit : {CLASS_NAMES[pred_label]}')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

# Recherche de la derniere couche conv du modele fine-tune
last_conv = [l.name for l in effnet_ft.layers if isinstance(l, tf.keras.layers.Conv2D)]
if not last_conv:
    # Chercher dans la base EfficientNet
    for l in effnet_ft.layers:
        if hasattr(l, 'layers'):
            conv_layers = [sl.name for sl in l.layers if isinstance(sl, tf.keras.layers.Conv2D)]
            if conv_layers:
                last_conv = conv_layers
                break

print(f"Derniere couche conv : {last_conv[-1] if last_conv else 'non trouvee'}")
LAST_CONV = last_conv[-1] if last_conv else 'top_conv'

In [ ]:
def get_saliency_map(model, img_array):
    """
    Carte de saillance : gradient de la prediction par rapport a l'image.
    Equivalent fonctionnel de Grad-CAM pour les modeles imbriques Keras 3.
    Met en evidence les regions de l'image qui ont le plus influence la decision.
    """
    img_t = tf.cast(np.expand_dims(img_array, 0), tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_t)
        pred = model(img_t, training=False)
        loss = pred[:, 0]

    grads   = tape.gradient(loss, img_t)
    heatmap = tf.reduce_max(tf.abs(grads[0]), axis=-1)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def display_saliency(model, img_array, true_label, pred_label):
    import cv2
    heatmap        = get_saliency_map(model, img_array)
    heatmap_res    = cv2.resize(heatmap, (224, 224))
    heatmap_col    = cm.jet(np.uint8(255 * heatmap_res))[:, :, :3]
    superimposed   = heatmap_col * 0.4 + img_array / 255.0
    superimposed  /= superimposed.max()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_array.astype('uint8'), cmap='gray')
    axes[0].set_title('Radiographie originale')
    axes[0].axis('off')
    axes[1].imshow(heatmap_res, cmap='jet')
    axes[1].set_title('Carte de saillance')
    axes[1].axis('off')
    axes[2].imshow(superimposed)
    axes[2].set_title(
        f"Reel : {CLASS_NAMES[true_label]}\nPredit : {CLASS_NAMES[pred_label]}"
    )
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()


# Affichage sur 2 NORMAL et 2 PNEUMONIA correctement classes
y_proba_all = effnet_ft.predict(X_test, batch_size=32, verbose=0).flatten()
y_pred_all  = (y_proba_all >= 0.3).astype(int)

for target_class, label in [(0, 'NORMAL'), (1, 'PNEUMONIA')]:
    correct_idxs = np.where((y_test == target_class) & (y_pred_all == target_class))[0]
    for idx in correct_idxs[:2]:
        print(f"Visualisation : {label}")
        display_saliency(
            effnet_ft, X_test[idx],
            true_label=y_test[idx],
            pred_label=y_pred_all[idx]
        )

---
## 9. Fiche de synthese comparative

**Fiche de synthese exigee par le projet :**

In [ ]:
synthesis = pd.DataFrame([
    {
        'Modele':             'CNN Custom (fait maison)',
        'Type':               'From scratch',
        'Params entrainables': '~6M',
        'Accuracy':           f"{results_cnn['accuracy']:.4f}",
        'AUC':                f"{results_cnn['auc']:.4f}",
        'Recall PNEUMONIA':   f"{results_cnn['recall']:.4f}",
        'F1':                 f"{results_cnn['f1']:.4f}",
        'Specificite':        f"{results_cnn['specificity']:.4f}",
        'Avantage':           'Leger, interpretable, personnalise',
        'Limite':             'Moins performant, besoin de plus de donnees'
    },
    {
        'Modele':             'EfficientNetB0 Frozen',
        'Type':               'Transfer Learning (base gelee)',
        'Params entrainables': '~300K',
        'Accuracy':           f"{results_eff1['accuracy']:.4f}",
        'AUC':                f"{results_eff1['auc']:.4f}",
        'Recall PNEUMONIA':   f"{results_eff1['recall']:.4f}",
        'F1':                 f"{results_eff1['f1']:.4f}",
        'Specificite':        f"{results_eff1['specificity']:.4f}",
        'Avantage':           'Rapide a entrainer, stable',
        'Limite':             'Features ImageNet pas optimales pour radio'
    },
    {
        'Modele':             'EfficientNetB0 Fine-tuned',
        'Type':               'Transfer Learning (affine)',
        'Params entrainables': '~2.8M',
        'Accuracy':           f"{results_eff2['accuracy']:.4f}",
        'AUC':                f"{results_eff2['auc']:.4f}",
        'Recall PNEUMONIA':   f"{results_eff2['recall']:.4f}",
        'F1':                 f"{results_eff2['f1']:.4f}",
        'Specificite':        f"{results_eff2['specificity']:.4f}",
        'Avantage':           'Meilleures performances, adapte au domaine',
        'Limite':             'Plus long a entrainer, risque overfitting'
    }
])

print("FICHE DE SYNTHESE COMPARATIVE")
print("=" * 80)
print(synthesis[['Modele','Type','Accuracy','AUC','Recall PNEUMONIA','F1','Specificite']].to_string(index=False))
print()

# Graphe comparatif
metrics_comp = ['accuracy','auc','recall','f1','specificity']
labels_comp  = ['Accuracy','AUC','Recall','F1','Specificite']
x = np.arange(len(labels_comp))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, (res, color) in enumerate(zip(all_results, ['steelblue','tomato','seagreen'])):
    vals = [res[m] for m in metrics_comp]
    bars = ax.bar(x + i*width, vals, width, label=res['model'], color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(labels_comp)
ax.set_ylim(0, 1.1)
ax.set_title('Comparaison des modeles : Test set officiel', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCONCLUSION :")
print("EfficientNetB0 Fine-tuned est le modele retenu pour la production.")
print("Il offre le meilleur rappel (recall) sur la classe PNEUMONIA, metrique prioritaire")
print("dans un contexte de diagnostic medical ou les faux negatifs sont inacceptables.")
print("Avec un seuil abaisse a 0.3, le recall est maximise tout en maintenant une")
print("specificite raisonnable pour eviter les faux positifs en exces.")

---
## 10. Test sur une nouvelle image

In [ ]:
def predict_xray(img_path, model, class_names, threshold=0.3):
    img = PILImage.open(img_path).convert('RGB').resize((224, 224))
    img_array = np.array(img, dtype='float32')

    proba = model.predict(np.expand_dims(img_array, 0), verbose=0)[0][0]
    label = class_names[int(proba >= threshold)]
    color = 'tomato' if label == 'PNEUMONIA' else 'steelblue'

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_array.astype('uint8'), cmap='gray')
    axes[0].axis('off')
    axes[0].set_title('Radiographie')

    axes[1].barh(['NORMAL', 'PNEUMONIA'], [1-proba, proba], color=['steelblue','tomato'])
    axes[1].set_xlim(0, 1)
    axes[1].axvline(threshold, color='gray', linestyle='--', label=f'Seuil={threshold}')
    axes[1].set_title(f'Diagnostic : {label}\nConfiance : {max(proba, 1-proba)*100:.1f}%',
                      color=color, fontweight='bold')
    axes[1].legend()
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

# Test sur les premieres images du test set officiel
for cls in ['NORMAL', 'PNEUMONIA']:
    sample_path = list((TEST_DIR / cls).glob('*.jpeg'))[0]
    print(f"\nTest : {cls}")
    predict_xray(sample_path, effnet_ft, CLASS_NAMES, threshold=0.3)

---
## 11. Sauvegarde

In [ ]:
import json

# Sauvegarde des 3 modeles
cnn_custom.save('cnn_custom_pneumonia.keras')
effnet_frozen.save('effnet_frozen_pneumonia.keras')
effnet_ft.save('effnet_finetuned_pneumonia.keras')

# Sauvegarde de la configuration de prediction (seuil, classes)
config = {
    'model_path':  'effnet_finetuned_pneumonia.keras',
    'class_names': CLASS_NAMES,
    'threshold':   0.3,
    'img_size':    list(IMG_SIZE),
    'note':        'EfficientNetB0 Fine-tuned, seuil optimise pour maximiser le recall PNEUMONIA'
}
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Sauvegardes :")
print("  cnn_custom_pneumonia.keras")
print("  effnet_frozen_pneumonia.keras")
print("  effnet_finetuned_pneumonia.keras  <- modele de production")
print("  model_config.json")

# Verification rechargement
model_prod = load_model('effnet_finetuned_pneumonia.keras')
test_pred  = (model_prod.predict(X_test[:5], verbose=0).flatten() >= 0.3).astype(int)
orig_pred  = (effnet_ft.predict(X_test[:5], verbose=0).flatten() >= 0.3).astype(int)
print(f"\nCoherence rechargement : {'OK' if np.all(test_pred == orig_pred) else 'ERREUR'}")